# 10 · INT8/INT4 quantization and SmoothQuant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/10_quantization.ipynb) &nbsp; 📖 Concept deck: [Quantization](https://nilesh-salpe.github.io/visual-ai/inference/07-quantization/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** symmetric quantization (per-tensor, per-channel, per-group), a W8A8 matmul, and SmoothQuant's outlier migration (Xiao et al., ICML 2023), with error measurements.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def rel_err(a, b): return float(np.linalg.norm(a - b) / np.linalg.norm(b))
rng0 = np.random.default_rng(0)
X = rng0.normal(0, 1, (64, 128)); X[:, [5, 77]] *= 40          # activations with two outlier channels (as observed in LLMs)
W = rng0.normal(0, 0.05, (128, 96))
print('X: activations [64 x 128] with outlier channels 5 and 77;  W: weights [128 x 96]')

## Task 1: Symmetric quantize/dequantize

`scale = max|x| / (2^{bits-1} − 1)`, `q = clip(round(x / scale), −qmax, qmax)`.
With `axis=None` use one scale for the whole tensor. Otherwise reduce over `axis` with `keepdims=True` (e.g. `axis=0` on W[in, out] gives one scale per **output channel**).
Return `(q, scale)`; dequantize with `q * scale`.

In [ ]:
def quantize_sym(x, bits=8, axis=None):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_quantize_sym(f):
    q, s = f(W, 8)
    assert np.abs(q).max() <= 127 and np.allclose(q, np.round(q)), "int8 values must be integers in [-127, 127]"
    assert np.abs(q * s - W).max() <= np.max(s) / 2 + 1e-12, "error must be at most half a step"
    q4, s4 = f(W, 4, axis=0)
    assert np.shape(s4) == (1, 96), f"per-output-channel scales should have shape (1, 96), got {np.shape(s4)}"
    assert np.abs(q4).max() <= 7
    Wm = W.copy(); Wm[:, 0] *= 30
    assert rel_err(f(Wm, 8, 0)[0] * f(Wm, 8, 0)[1], Wm) < rel_err(f(Wm, 8)[0] * f(Wm, 8)[1], Wm), "per-channel should beat per-tensor when channel ranges differ"

check("quantize_sym", _test_quantize_sym, quantize_sym)

## Task 2: A W8A8 matmul

Quantize **X per-tensor** and **W per-output-channel** to INT8, multiply the integer matrices (use `np.int64` to stand in for INT32 accumulation), then rescale to float: `Y ≈ (qX @ qW) * sX * sW`.

In [ ]:
def w8a8_matmul(X, W, quant=None):
    quant = quant or quantize_sym
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_w8a8_matmul(f):
    Xs = rng0.normal(0, 1, (16, 128)); ref = Xs @ W
    assert rel_err(f(Xs, W), ref) < 0.02, "W8A8 on well-behaved activations should be within ~2% of FP"
    assert rel_err(f(X, W), X @ W) > 0.02, "(sanity) outlier activations should hurt per-tensor W8A8"

check("w8a8_matmul", _test_w8a8_matmul, w8a8_matmul)

## Task 3: SmoothQuant: migrate outliers into the weights

Per input channel $j$: $s_j = \max|X_{:,j}|^{\alpha} / \max|W_{j,:}|^{1-\alpha}$ (paper eq. 4, α = 0.5 by default).
Return `(X / s, W * s[:, None], s)`. The product is **mathematically unchanged**: $(X\,\mathrm{diag}(s)^{-1})(\mathrm{diag}(s)\,W) = XW$.

In [ ]:
def smoothquant(X, W, alpha=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _w8a8_ref(A, B):
    qa = lambda x, ax: (lambda s: (np.clip(np.round(x / s), -127, 127), s))(np.maximum(np.abs(x).max(axis=ax, keepdims=ax is not None), 1e-12) / 127)
    (x8, sx), (w8, sw) = qa(A, None), qa(B, 0)
    return (x8.astype(np.int64) @ w8.astype(np.int64)) * sx * sw

def _test_smoothquant(f):
    Xs, Ws, s = f(X, W, 0.5)
    assert s.shape == (128,), f"one factor per input channel: shape {s.shape}"
    assert np.allclose(Xs @ Ws, X @ W, atol=1e-9), "smoothing must not change X @ W"
    before = rel_err(_w8a8_ref(X, W), X @ W); after = rel_err(_w8a8_ref(Xs, Ws), X @ W)
    assert after < 0.5 * before, f"smoothing should at least halve W8A8 error (before {before:.3f}, after {after:.3f})"

check("smoothquant", _test_smoothquant, smoothquant)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Symmetric quantize/dequantize
def quantize_sym_solution(x, bits=8, axis=None):
    qmax = 2 ** (bits - 1) - 1
    amax = np.abs(x).max() if axis is None else np.abs(x).max(axis=axis, keepdims=True)
    scale = np.maximum(amax, 1e-12) / qmax
    return np.clip(np.round(x / scale), -qmax, qmax), scale

check("quantize_sym (reference)", _test_quantize_sym, quantize_sym_solution)

In [ ]:
# @title Solution for Task 2: A W8A8 matmul
def w8a8_matmul_solution(X, W, quant=None):
    quant = quant or quantize_sym_solution
    qx, sx = quant(X, 8)               # per-tensor activations
    qw, sw = quant(W, 8, axis=0)       # per-output-channel weights, shape (1, out)
    return (qx.astype(np.int64) @ qw.astype(np.int64)) * sx * sw

check("w8a8_matmul (reference)", _test_w8a8_matmul, w8a8_matmul_solution)

In [ ]:
# @title Solution for Task 3: SmoothQuant: migrate outliers into the weights
def smoothquant_solution(X, W, alpha=0.5):
    s = np.abs(X).max(axis=0) ** alpha / np.maximum(np.abs(W).max(axis=1), 1e-12) ** (1 - alpha)
    return X / s, W * s[:, None], s

check("smoothquant (reference)", _test_smoothquant, smoothquant_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
print(f"W8A8 error, raw outlier activations : {rel_err(w8a8_matmul_solution(X, W), X @ W):.3%}")
for a in (0.25, 0.5, 0.75, 0.9):
    Xs, Ws, _ = smoothquant_solution(X, W, a)
    print(f"W8A8 error after SmoothQuant a={a:<4}: {rel_err(w8a8_matmul_solution(Xs, Ws), X @ W):.3%}")
for g in (128, 32, 8):
    Wg = W.reshape(128 // g, g, 96); q, s = quantize_sym_solution(Wg, 4, axis=1)
    print(f"INT4 weight-only, group size {g:3d}: weight error {rel_err((q * s).reshape(128, 96), W):.2%}")

---
## Takeaways
- Quantization error scales with the **range** each scale must cover. Per-channel and per-group scales shrink that range.
- LLM activations have **outlier channels** that break per-tensor INT8. SmoothQuant moves that difficulty into the weights with an exactly equivalent rescaling (**up to 1.56× faster, 2× less memory** reported).
- Weight-only methods (GPTQ, AWQ) push weights to 3–4 bits. AWQ protects the ~1% of salient weights identified from activation statistics.